In [3]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Some global variables

In [4]:
filter_start = "2025-10-01"
filter_end = "2026-10-01"

period_start = "2023-10-01"
period_end = "2026-10-01"

In [26]:
# Set up exchange rate dataframe for every day in the period

exchange_rate = yf.Ticker("CAD=X").history(start = period_start, end = period_end)["Close"]

# Keep only year month date, datetime objects, in index column of dataframe
exchange_rate.index = pd.to_datetime(exchange_rate.index.strftime('%Y-%m-%d'))

display(exchange_rate.head())

Date
2023-10-02    1.35737
2023-10-03    1.36763
2023-10-04    1.37091
2023-10-05    1.37442
2023-10-06    1.37057
Name: Close, dtype: float64

Extracting stocks

In [5]:
# EXTRACTING TICKERS. MAKE SURE TO CHANGE FILE NAME FOR ACTUAL SUBMISSION
tickers = pd.read_csv("Tickers.csv", header=None)[0].to_list()
close_prices = pd.DataFrame()

for tick in tickers:
    t = yf.Ticker(tick)
    information = t.info

    # Check if there is data in here (i.e. not a delisted or invalid ticker)
    if t.history(period='max').empty:
        print(tick, "doesn't exist")
        continue
    
    # Check if it's a CAD or USD stock
    if (information["market"] != "ca_market") and (information["market"] != "us_market"):
        print(tick, "wrong market")
        continue

    # Check if it's a stock and not something like a bond or index fund
    if information["quoteType"] != "EQUITY":
        print(tick, "not stock")
        continue

    if (information["currency"] != "CAD") and (information["currency"] != "USD"):
        print(tick, "currency wrong")
        continue
    

    temp = yf.download(tick, filter_start, filter_end)

    if temp["Volume"].mean()[tick] < 5000:
        print(tick, "not enough volume")
        continue
    
    close_prices[tick] = temp["Close"]

# Filters out the first and last trading day of each month
idx = close_prices.index.to_series()
date_series_first=idx.groupby([idx.dt.year, idx.dt.month]).first()
date_series_last=idx.groupby([idx.dt.year, idx.dt.month]).last()
monthly_first = []
monthly_last = []
for i in date_series_first:
    monthly_first.append(i)
for i in date_series_last:
    monthly_last.append(i)

for i in range(len(monthly_first)):
    start_ind = close_prices.index.get_loc(monthly_first[i])
    end_ind = close_prices.index.get_loc(monthly_last[i])

    if (end_ind - start_ind + 1) < 18:
        close_prices.drop(close_prices.loc[monthly_first[i]:monthly_last[i]].index)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$AGN: possibly delisted; no timezone found


AGN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$BK: possibly delisted; no price data found  (1d 1927-12-15 -> 2026-11-20)


BK doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$CELG: possibly delisted; no timezone found


CELG doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$MON: possibly delisted; no timezone found


MON doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$RTN: possibly delisted; no timezone found


RTN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [25]:
display(close_prices.head())

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,254.316589,235.580246,129.656555,236.889038,77.725349,220.630005,324.080414,215.199997,49.622257,6.67,...,65.924820,163.093613,208.619995,20.261112,107.826408,174.982269,339.266205,229.894775,78.023796,46.106377
2025-10-02,255.989151,228.041824,129.190262,237.501389,77.725349,222.410004,325.974274,217.429993,49.426426,6.63,...,68.169777,165.405457,211.240005,19.976393,106.874413,176.854416,344.545593,229.679321,79.124161,46.135265
2025-10-03,256.875214,225.487259,130.744537,238.453979,79.072174,219.509995,325.737518,216.300003,49.583092,6.28,...,68.486244,165.728729,224.889999,20.104977,108.913048,174.914383,350.857544,231.912247,80.243004,46.289337
2025-10-06,255.551132,221.901215,129.918823,241.224197,81.024086,220.899994,327.463654,219.729996,49.338310,6.68,...,70.503761,165.180145,229.520004,19.893730,108.951515,176.359711,349.464630,232.509659,80.095062,45.952305
2025-10-07,255.342041,224.446136,129.219421,244.198547,82.029327,221.779999,323.498474,221.820007,49.240398,6.48,...,73.787132,162.084625,225.100006,19.875360,108.605331,171.742432,354.227814,227.074219,79.503265,45.740452


In [35]:
full_data = pd.DataFrame()

for tick in close_prices:
    prices = yf.download(tick, period_start, period_end)["Close"]
    
    if yf.Ticker(tick).info["currency"] == "CAD":
        full_data[tick] = prices[tick]
    else:
        full_data[tick] = prices[tick] * exchange_rate

full_data["CBIL.TO"] = yf.download("CBIL.TO", period_start, period_end, auto_adjust=True)["Close"]

full_data = full_data

display(full_data.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,CBIL.TO
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,232.519125,180.319242,121.325736,393.012968,76.241152,175.725132,195.350744,254.954813,33.728010,6.02,...,141.522601,73.839996,17.100687,70.040123,198.282212,655.916088,257.465562,174.719966,38.394417,45.243137
2023-10-03,232.456366,180.677288,122.319412,393.376058,75.013732,170.570816,190.994175,258.372658,32.977460,5.98,...,141.424391,70.440002,17.204714,69.378128,196.408580,654.324020,260.871020,174.530851,37.721729,45.229584
2023-10-04,234.716886,181.429996,122.433557,398.388477,76.116891,174.105576,193.537707,255.990027,33.094829,5.86,...,141.326150,72.230003,17.756866,69.195213,197.781094,657.373793,259.558983,174.915339,37.691549,45.243137
2023-10-05,237.011707,181.598919,123.452841,399.757000,77.327370,173.121948,195.441646,256.040701,33.333052,5.20,...,141.623504,71.510002,18.100962,69.997536,194.697581,666.297766,258.126554,174.975326,38.550104,45.247662
2023-10-06,239.834014,182.060483,123.977218,401.894164,78.033783,175.378129,197.848890,256.817403,33.252432,5.15,...,142.615890,73.269997,18.044949,70.359039,196.517888,675.474447,259.738376,175.497617,38.430051,45.256702


Sharpe calculation

In [8]:
half_life = 100

In [9]:
def get_stats(tick, half_life):
    s = full_data[tick].dropna()
    delta = np.log(1 + s.pct_change()).ewm(halflife = half_life)
    standard_deviation = delta.std().iloc[-1]
    average = delta.mean().iloc[-1]

    return average, standard_deviation

In [10]:
Rf = get_stats("CBIL.TO", half_life)[0]
print("Risk free return:", Rf)

sharpe = {}

for tick in close_prices:
    stats = get_stats(tick, half_life)
    sharpe[tick] = (stats[0] - Rf)/stats[1]

sharpe = dict(sorted(sharpe.items(), key = lambda item:item[1]))
print(sharpe)

Risk free return: 9.551858949783563e-05
{'T.TO': np.float64(-0.1370941701032326), 'PEP': np.float64(-0.06620120250492213), 'BA': np.float64(-0.04676056633679974), 'ABT': np.float64(-0.0358478049822761), 'UPS': np.float64(-0.030357459658023046), 'AXP': np.float64(-0.028203824390032932), 'ACN': np.float64(-0.025693648583782917), 'LMT': np.float64(-0.016877787198906657), 'CL': np.float64(-0.016805227645212452), 'AIG': np.float64(-0.01659369877850492), 'PYPL': np.float64(-0.011953122955851788), 'UNH': np.float64(-0.0017901593498641497), 'BLK': np.float64(0.002384412437587984), 'BAC': np.float64(0.007459434221443712), 'AMZN': np.float64(0.01296985656956304), 'QCOM': np.float64(0.018208411607913182), 'UNP': np.float64(0.022649413855545752), 'MO': np.float64(0.025285205114700886), 'USB': np.float64(0.029153806293359005), 'SHOP.TO': np.float64(0.03248731983922844), 'LLY': np.float64(0.037825593357357926), 'BMY': np.float64(0.03893588397480373), 'PM': np.float64(0.042941808568967045), 'C': np.f

In [11]:
def weighted_corr(t1, t2):
    d1 = full_data[t1].pct_change(fill_method=None)
    d2 = full_data[t2].pct_change(fill_method=None)
    return d1.ewm(halflife=half_life).corr(d2).dropna().iloc[-1]

Correlation matrix

In [12]:
corr_mat = pd.DataFrame()

for t1 in close_prices:
    for t2 in close_prices:
        corr_mat.loc[t1, t2] = weighted_corr(t1, t2)

In [13]:
display(corr_mat)

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
AAPL,1.000000,0.169150,0.098866,0.180679,0.229776,0.111151,0.246738,0.237700,0.251212,0.022890,...,0.269392,0.214442,0.261143,0.326863,0.223009,0.135810,-0.035881,0.108889,0.215455,0.267526
ABBV,0.169150,1.000000,0.321891,0.187360,0.194110,-0.155674,0.070051,-0.036207,0.075035,-0.015386,...,0.138632,-0.094889,0.016319,0.113246,0.022721,-0.077463,0.135933,0.199029,0.160271,0.138871
ABT,0.098866,0.321891,1.000000,0.285273,0.222522,-0.034579,0.111076,0.081138,0.135495,-0.225666,...,0.169627,-0.137684,0.157454,0.099337,0.000159,-0.144920,0.091165,0.256965,0.141354,0.194551
ACN,0.180679,0.187360,0.285273,1.000000,0.213496,0.088046,0.269786,0.095374,0.107067,0.027500,...,0.301324,-0.026642,0.385755,0.288612,-0.037426,-0.257377,0.074579,0.076987,0.120719,0.213970
AIG,0.229776,0.194110,0.222522,0.213496,1.000000,0.015158,0.185538,0.147174,0.244671,-0.100936,...,0.076437,-0.033823,0.122276,0.052265,0.147041,-0.012408,0.099624,0.288567,0.119157,0.258253
AMZN,0.111151,-0.155674,-0.034579,0.088046,0.015158,1.000000,0.322887,0.273834,0.254508,0.134281,...,0.210283,0.170858,0.226590,-0.286972,0.224261,0.133083,0.038342,0.053068,0.123004,0.255499
AXP,0.246738,0.070051,0.111076,0.269786,0.185538,0.322887,1.000000,0.383087,0.598730,0.158492,...,0.305911,0.202895,0.370173,0.081958,0.429763,0.152980,0.193191,0.180926,0.281189,0.666731
BA,0.237700,-0.036207,0.081138,0.095374,0.147174,0.273834,0.383087,1.000000,0.323247,0.045766,...,0.205021,0.276180,0.184316,0.044927,0.306190,0.263537,0.110285,0.069643,0.235342,0.320842
BAC,0.251212,0.075035,0.135495,0.107067,0.244671,0.254508,0.598730,0.323247,1.000000,0.092415,...,0.165409,0.189872,0.146564,-0.028299,0.505531,0.165842,0.115228,0.207871,0.264538,0.743015
BB.TO,0.022890,-0.015386,-0.225666,0.027500,-0.100936,0.134281,0.158492,0.045766,0.092415,1.000000,...,0.071328,0.218934,0.181671,-0.082448,0.224928,0.205908,0.045081,0.038184,0.156753,0.137061
